# Défi quotidien — Analyse textuelle des livres de Lewis Carroll

**Objectifs :** prétraitement du texte, analyse de texte, sac de mots (BoW) et TF-IDF, nuages de mots.

**Corpus (3 livres de Lewis Carroll, Project Gutenberg) :**
1. *Alice's Adventures in Wonderland* (Les aventures d'Alice au pays des merveilles) — #11
2. *Through the Looking-Glass* (De l'autre côté du miroir) — #12
3. *A Tangled Tale* (Une histoire complexe) — #29042


## 0. Environnement et installation

> **Conseil du TP :** créez un environnement virtuel dédié au cours de NLP.
> ```bash
> python -m venv nlp-env
> source nlp-env/bin/activate      # Windows : nlp-env\\Scripts\\activate
> pip install nltk spacy gensim scikit-learn wordcloud matplotlib requests
> python -m spacy download en_core_web_sm
> ```
> Sur Google Colab, exécutez simplement la cellule ci-dessous.

In [ ]:
# Colab : décommentez si besoin
# !pip install -q nltk spacy gensim scikit-learn wordcloud matplotlib requests
# !python -m spacy download en_core_web_sm

In [ ]:
import re
import string
import requests
import numpy as np
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

import spacy
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Ressources NLTK
for res in ['punkt', 'punkt_tab', 'stopwords',
            'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng',
            'maxent_ne_chunker', 'maxent_ne_chunker_tab', 'words']:
    nltk.download(res, quiet=True)

# Modèle spaCy
nlp = spacy.load('en_core_web_sm')
print("Environnement prêt ✅")

# Partie 1 — Prétraitement du texte

## 1.1 Fonction `load_texts()`

Elle reçoit une liste d'URL, télécharge chaque livre, supprime les non-lettres via **regex**, et renvoie le corpus nettoyé.

On coupe aussi l'en-tête et le pied de page de Project Gutenberg en se basant sur les repères **START** / **END** (équivalents de « DÉBUT » / « FIN »), afin d'ignorer les mentions légales et l'auteur.

In [ ]:
BOOK_URLS = [
    'https://www.gutenberg.org/files/11/11-0.txt',      # Alice's Adventures in Wonderland
    'https://www.gutenberg.org/files/12/12-0.txt',      # Through the Looking-Glass
    'https://www.gutenberg.org/files/29042/29042-0.txt' # A Tangled Tale
]
TITLES = ["Alice's Adventures in Wonderland",
          "Through the Looking-Glass",
          "A Tangled Tale"]

def load_texts(urls):
    """Télécharge et nettoie une liste de livres, renvoie le corpus (liste de str)."""
    corpus = []
    headers = {'User-Agent': 'Mozilla/5.0'}   # évite d'éventuels blocages
    for url in urls:
        raw = requests.get(url, headers=headers, timeout=30).text

        # --- couper l'en-tête / pied de page Gutenberg (START ... END) ---
        start = raw.find('*** START')
        end = raw.find('*** END')
        if start != -1:
            start = raw.find('\n', start)          # aller à la fin de la ligne START
        if start != -1 and end != -1:
            raw = raw[start:end]

        # --- nettoyage regex : garder uniquement lettres et espaces ---
        cleaned = re.sub(r'[^A-Za-z\s]', ' ', raw)
        cleaned = re.sub(r'\s+', ' ', cleaned).strip()
        corpus.append(cleaned)
    return corpus

corpus = load_texts(BOOK_URLS)
print("Nombre de livres chargés :", len(corpus))

## 1.2 Aperçu : 200 premiers caractères de chaque texte

In [ ]:
for title, text in zip(TITLES, corpus):
    print(f"===== {title} =====")
    print(text[:200], "\n")

**Parties non pertinentes ?** Oui : l'en-tête et le pied de page de Project Gutenberg (licence, nom de l'auteur, table des matières) ne concernent pas le contenu. Ils ont déjà été retirés dans `load_texts()` grâce au découpage **START / END**. Le nettoyage regex a par ailleurs supprimé la ponctuation et les chiffres.

## 1.3 Tokenisation — 150 premiers tokens de chaque livre

In [ ]:
tokens = [word_tokenize(text.lower()) for text in corpus]

for title, toks in zip(TITLES, tokens):
    print(f"===== {title} ({len(toks)} tokens) =====")
    print(toks[:150], "\n")

## 1.4 Suppression des mots vides (NLTK)

On vérifie ensuite avec `count()` que des stopwords comme *i, me, my, we, our…* ont bien disparu.

In [ ]:
stop_words = set(stopwords.words('english'))
tokens_no_stop = [[w for w in toks if w not in stop_words] for toks in tokens]

check_words = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves']
for title, before, after in zip(TITLES, tokens, tokens_no_stop):
    print(f"===== {title} =====")
    for w in check_words:
        print(f"  '{w}': avant={before.count(w):>4} | après={after.count(w)}")
    print(f"  tokens: {len(before)} -> {len(after)}\n")

## 1.5 Racinisation (stemming) avec `PorterStemmer` — 50 premiers tokens

In [ ]:
stemmer = PorterStemmer()
tokens_stemmed = [[stemmer.stem(w) for w in toks] for toks in tokens_no_stop]

for title, stems in zip(TITLES, tokens_stemmed):
    print(f"===== {title} =====")
    print(stems[:50], "\n")

## 1.6 Lemmatisation avec spaCy (`en_core_web_sm`) — 50 premiers tokens

Dans spaCy, le lemme est accessible via l'attribut `token.lemma_`.

In [ ]:
tokens_lemmatized = []
for toks in tokens_no_stop:
    # on limite l'entrée pour la rapidité ; ajustez si besoin
    doc = nlp(" ".join(toks[:500]))
    tokens_lemmatized.append([t.lemma_ for t in doc])

for title, lemmas in zip(TITLES, tokens_lemmatized):
    print(f"===== {title} =====")
    print(lemmas[:50], "\n")

## 1.7 Analyse : lemmatisation vs racinisation

- La **racinisation** (Porter) coupe mécaniquement les suffixes selon des règles. Résultat rapide mais souvent **non lexical** : *adventures → adventur*, *little → littl*, *happily → happili*.
- La **lemmatisation** (spaCy) ramène chaque mot à sa **forme canonique réelle** (le lemme du dictionnaire) en s'appuyant sur la morphologie et la catégorie grammaticale : *adventures → adventure*, *better → good*, *was → be*.
- **Pourquoi cette différence ?** Le stemming est une heuristique de troncature, sans dictionnaire ni contexte. La lemmatisation utilise un lexique et l'analyse morphosyntaxique, ce qui la rend plus précise mais plus lente. On privilégie le stemming pour la vitesse, la lemmatisation pour la qualité/lisibilité.

## 1.8 Étiquetage POS (NLTK)

In [ ]:
def perform_pos_tagging(tokens_list):
    return nltk.pos_tag(tokens_list)

for title, toks in zip(TITLES, tokens):
    pos = perform_pos_tagging(toks[:100])   # 100 premiers pour l'affichage
    print(f"===== {title} — POS (100 premiers) =====")
    print(pos, "\n")

## 1.9 Entités nommées (NLTK)

In [ ]:
def extract_entities(text_tokens):
    """Renvoie les entités nommées via nltk.ne_chunk."""
    pos = nltk.pos_tag(text_tokens)
    tree = nltk.ne_chunk(pos)
    entities = []
    for subtree in tree:
        if hasattr(subtree, 'label'):
            entity = " ".join(word for word, tag in subtree.leaves())
            entities.append((entity, subtree.label()))
    return entities

# NLTK NER fonctionne mieux sur le texte original (avec majuscules).
# On ré-extrait un échantillon depuis le texte nettoyé mais NON mis en minuscules.
for title, text in zip(TITLES, corpus):
    sample_tokens = word_tokenize(text[:3000])   # échantillon (garde les majuscules)
    ents = extract_entities(sample_tokens)
    print(f"===== {title} — entités (échantillon) =====")
    print(ents[:20], "\n")

> **Remarque :** la reconnaissance d'entités de NLTK s'appuie fortement sur les **majuscules**. Comme la mise en minuscules détruit cet indice, on applique le NER sur un échantillon de texte conservant la casse d'origine.

# Partie 2 — Analyse du texte

## 2.1 Nuage de mots pour chaque livre

On utilise le texte **racinisé et sans stopwords** (le plus propre pour compter les thèmes).

In [ ]:
stemmed_docs = [" ".join(stems) for stems in tokens_stemmed]

fig, axes = plt.subplots(1, 3, figsize=(22, 8))
for ax, title, doc in zip(axes, TITLES, stemmed_docs):
    wc = WordCloud(width=800, height=600, background_color='white',
                   colormap='viridis', max_words=100).generate(doc)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=14)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2.2 BoW — les 5 mots les plus fréquents dans tout le corpus

**Quel texte prétraité utiliser ?** Le texte **racinisé et sans mots vides** : on évite ainsi de compter les stopwords et on regroupe les variantes d'un même mot (*say/says/said → say*), ce qui donne des comptes plus fiables que sur le texte brut ou seulement tokenisé.

In [ ]:
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(stemmed_docs)
vocab = vectorizer.get_feature_names_out()

total_freq = np.asarray(bow_matrix.sum(axis=0)).ravel()
top5_idx = np.argsort(total_freq)[::-1][:5]

print("5 mots les plus fréquents (tout le corpus) :")
for i in top5_idx:
    print(f"  {vocab[i]:<12} : {int(total_freq[i])}")

## 2.3 Lecture du BoW : n° de document, index du mot, nombre d'occurrences

La matrice BoW est de forme `(n_documents, n_mots)`. Chaque valeur `matrice[doc, index] = fréquence`.

In [ ]:
print("Forme de la matrice BoW (documents, mots) :", bow_matrix.shape)
print("\nExtrait de la représentation creuse (doc_index, mot_index) -> occurrences :\n")

# Afficher quelques entrées non nulles interprétées
coo = bow_matrix.tocoo()
shown = 0
for doc_i, word_i, count in zip(coo.row, coo.col, coo.data):
    print(f"  Document n°{doc_i} ('{TITLES[doc_i]}') | index {word_i} = '{vocab[word_i]}' | trouvé {count} fois")
    shown += 1
    if shown >= 10:
        break

# Exemple ciblé : le mot 'alic' dans chaque document
if 'alic' in vocab:
    wi = list(vocab).index('alic')
    print(f"\nMot 'alic' (index {wi}) par document :")
    for di in range(bow_matrix.shape[0]):
        print(f"  Document n°{di} ({TITLES[di]}) : {bow_matrix[di, wi]} fois")

## 2.4 Diagramme circulaire des 5 mots les plus fréquents (BoW)

In [ ]:
labels = [vocab[i] for i in top5_idx]
sizes = [int(total_freq[i]) for i in top5_idx]

plt.figure(figsize=(8, 8))
plt.pie(sizes, labels=[f"{l} ({s})" for l, s in zip(labels, sizes)],
        autopct='%1.1f%%', startangle=140)
plt.title("5 mots les plus fréquents (BoW) — tout le corpus")
plt.axis('equal')
plt.show()

## 2.5 Analyse des résultats BoW

Les mots les plus fréquents sont typiquement *said, alice, one, little, look*. Ils sont **attendus mais peu informatifs** : *said* et *one* sont des mots très courants dans tout roman, et *alice* est évidemment omniprésent dans deux des trois livres. Ils décrivent le **style narratif** plutôt que le **contenu spécifique** de chaque livre. C'est précisément le problème que TF-IDF va corriger.

# Partie 3 — Résoudre le problème de fréquence avec TF-IDF

TF-IDF pondère chaque mot par sa rareté à travers les documents : un mot fréquent dans **un** livre mais rare dans les **autres** obtient un score élevé (il est discriminant), tandis qu'un mot présent partout (comme *said*) est pénalisé.

## 3.1 BoW avec `TfidfVectorizer`

In [ ]:
# min_df=1, max_df=2 comme conseillé (petit corpus de 3 documents)
tfidf = TfidfVectorizer(min_df=1, max_df=2)
tfidf_matrix = tfidf.fit_transform(stemmed_docs)
tfidf_vocab = tfidf.get_feature_names_out()

print("Forme de la matrice TF-IDF :", tfidf_matrix.shape)

# Top 5 mots TF-IDF par document
top_words_per_doc = []
for di, title in enumerate(TITLES):
    row = tfidf_matrix[di].toarray().ravel()
    idx = np.argsort(row)[::-1][:5]
    words = [(tfidf_vocab[i], round(float(row[i]), 4)) for i in idx]
    top_words_per_doc.append(idx)
    print(f"\n{title} — top 5 TF-IDF :")
    for w, s in words:
        print(f"  {w:<12} : {s}")

## 3.2 Diagrammes circulaires des 5 mots les plus pertinents (TF-IDF) par document

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, di, title in zip(axes, range(len(TITLES)), TITLES):
    row = tfidf_matrix[di].toarray().ravel()
    idx = np.argsort(row)[::-1][:5]
    labels = [tfidf_vocab[i] for i in idx]
    sizes = [row[i] for i in idx]
    ax.pie(sizes, labels=[f"{l}\n{s:.3f}" for l, s in zip(labels, sizes)],
           autopct='%1.1f%%', startangle=140)
    ax.set_title(f"{title}\n(TF-IDF top 5)", fontsize=12)
    ax.axis('equal')
plt.tight_layout()
plt.show()

## 3.3 Analyse comparative BoW vs TF-IDF

- **BoW** faisait remonter des mots génériques (*said, one, alice*) — communs à tous les livres, donc peu discriminants.
- **TF-IDF** fait ressortir des mots **spécifiques et informatifs** à chaque livre :
  - *Alice au pays des merveilles* → personnages/éléments propres (gryphon, rabbit, hatter, mock turtle…)
  - *De l'autre côté du miroir* → humpty, dumpty, tweedledum, knight, kitten…
  - *Une histoire complexe* (livre de puzzles mathématiques) → clara, balbus, train, problem, solution…
- **Conclusion :** TF-IDF identifie bien l'« empreinte » thématique de chaque document en atténuant les mots omniprésents. C'est un bien meilleur descripteur du contenu propre à chaque livre que le simple comptage BoW.